<div style="font-family: 'Times New Roman', Georgia, serif; background-color: #fefdf8; padding: 20px; border: 2px solid #d4c4a8; border-radius: 8px; box-shadow: 0 4px 8px rgba(0,0,0,0.1); line-height: 1.6; max-width: 100%; box-sizing: border-box; overflow: hidden;">

<h3 style="color: #5d4e37; text-align: center; margin-bottom: 20px; font-weight: bold; max-width: 100%; word-wrap: break-word; hyphens: auto;">Loading ORIGINAL DATA</h3>

</div>

In [1]:
import numpy as np
from types import SimpleNamespace
import pandas as pd
from h5_utils import save

In [2]:
#DVL DATA LOADING   

data = np.loadtxt('ORIGINAL_DATA/1654015516_dvl_0.csv', delimiter=';')
RAW_DVL = SimpleNamespace()         # a struct-like container for the DVL data

RAW_DVL.ABS_TIME     = data[:, 0]   # Absolute timestamp
RAW_DVL.DT1          = data[:, 3]   # Delta‐time measurement 1
RAW_DVL.DT2          = data[:, 4]   # Delta‐time measurement 2
RAW_DVL.U            = data[:, 5]   # Forward velocity
RAW_DVL.V            = data[:, 6]   # Sway velocity 
RAW_DVL.W            = data[:, 7]   # Heave velocity
RAW_DVL.FOM          = data[:, 8]   # Figure‐of‐merit (quality of velocity fix)
RAW_DVL.DIS_BEAM_1   = data[:, 9]   # Distance from beam 1
RAW_DVL.DIS_BEAM_2   = data[:,10]   # Distance from beam 2
RAW_DVL.DIS_BEAM_3   = data[:,11]   # Distance from beam 3
RAW_DVL.DIS_BEAM_4   = data[:,12]   # Distance from beam 4
RAW_DVL.TENS_BATT    = data[:,13]   # Tension battery voltage
RAW_DVL.SOUND_SPEED  = data[:,14]   # Speed of sound used by DVL
RAW_DVL.DEPTH        = data[:,15]   # Depth measurement
RAW_DVL.WATER_TEMPER = data[:,16]   # Water temperature
RAW_DVL.STATUS       = data[:,17]   # DVL status
print('DVL data loaded successfully.')

# ECHOSOUNDER DATA LOADING   

data = np.loadtxt('ORIGINAL_DATA/1654015516_echoSounder_0.csv', delimiter=';')
RAW_ECHO = SimpleNamespace()

RAW_ECHO.ABS_TIME     = data[:, 0]     # absolute timestamp
RAW_ECHO.DISTANCE     = data[:, 2]     # measured distance
RAW_ECHO.WATER_TEMPER = data[:, 3]     # water temperature
RAW_ECHO.QUAT   = data[:, 4:8]          # quaternion [w, x, y, z] components
print('ECHOSOUNDER data loaded successfully.')

# IMU DATA LOADING   

# Euler
euler_data = np.loadtxt('ORIGINAL_DATA/1654015516_imu_euler_0.csv',delimiter=';')
RAW_IMU_EULER = SimpleNamespace()
RAW_IMU_EULER.ABS_TIME = euler_data[:, 0]       # absolute timestamp
RAW_IMU_EULER.EULER_ANGLE = euler_data[:, 2:5]  # euler angles [roll, pitch, yaw] in degrees

# Quaternion
quat_data = np.loadtxt('ORIGINAL_DATA/1654015516_imu_quat_0.csv',delimiter=';',)
RAW_IMU_QUAT = SimpleNamespace()
RAW_IMU_QUAT.ABS_TIME = quat_data[:, 0]  # absolute timestamp
RAW_IMU_QUAT.QUAT = quat_data[:, 2:6]    # quaternion [w, x, y, z] components
print('IMU data loaded successfully.')

# VERTICAL PROFILO (SEAKING) DATA LOADING 

df0 = pd.read_csv('ORIGINAL_DATA/1654015516_profilo_0.csv', sep = r';|\s+', engine='python', header=None)
df1 = pd.read_csv('ORIGINAL_DATA/1654015516_profilo_1.csv', sep = r';|\s+', engine='python', header=None)
df2 = pd.read_csv('ORIGINAL_DATA/1654015516_profilo_2.csv', sep = r';|\s+', engine='python', header=None)
Data_Profilo = pd.concat([df0, df1, df2], ignore_index=True).to_numpy()

RAW_PROFILO_SEAKING = SimpleNamespace()
RAW_PROFILO_SEAKING.ABS_TIME = Data_Profilo[:, 0]    # absolute timestamp
RAW_PROFILO_SEAKING.ANGLE    = Data_Profilo[:, 3] * (np.pi / 200.0)  # beam angle
RAW_PROFILO_SEAKING.RANGE    = Data_Profilo[:, 4]    # measured range
RAW_PROFILO_SEAKING.NBR_BINS = Data_Profilo[:, 5]    # number of bins
RAW_PROFILO_SEAKING.SKYLINE  = Data_Profilo[:, 6:]   # skyline data across bins
print('Vertical Profilo (Seaking) profiling data loaded successfully.')

# PING360 PROFILO DATA LOADING

ping_0 = pd.read_csv('ORIGINAL_DATA/1654015516_ping360_0.csv', sep = r';', engine='python', header=None)
ping_1 = pd.read_csv('ORIGINAL_DATA/1654015516_ping360_1.csv', sep = r';', engine='python', header=None)
ping_2 = pd.read_csv('ORIGINAL_DATA/1654015516_ping360_2.csv', sep = r';', engine='python', header=None)
ping_3 = pd.read_csv('ORIGINAL_DATA/1654015516_ping360_3.csv', sep = r';', engine='python', header=None)
ping_4 = pd.read_csv('ORIGINAL_DATA/1654015516_ping360_4.csv', sep = r';', engine='python', header=None)
Data_Profilo = pd.concat([ping_0, ping_1, ping_2, ping_3, ping_4], ignore_index=True).to_numpy()

RAW_PROFILO_PING360 = SimpleNamespace()
RAW_PROFILO_PING360.ABS_TIME   = Data_Profilo[:, 0]    # absolute timestamp
RAW_PROFILO_PING360.MODE       = Data_Profilo[:, 1]    # operating mode
RAW_PROFILO_PING360.GAIN       = Data_Profilo[:, 2]    # gain setting
RAW_PROFILO_PING360.ANGLE      = Data_Profilo[:, 4] * (np.pi / 200.0)   # beam angle
RAW_PROFILO_PING360.NBR_SCANS  = Data_Profilo[:, 8]    # number of scans
RAW_PROFILO_PING360.NBR_BINS   = Data_Profilo[:, 9]    # number of bins

# Fix the SKYLINE parsing manually
skyline_strings = Data_Profilo[:, 10]
processed_skyline = []
max_len = 0

for row in skyline_strings:
    if isinstance(row, bytes):
        row = row.decode('utf-8')
    else:
        row = str(row)
    
    numeric_row = np.array(row.split(','), dtype=float) # Split the string by commas and cast to float
    processed_skyline.append(numeric_row)
    if len(numeric_row) > max_len:
        max_len = len(numeric_row)
        
# Pad jagged rows with 0.0 to create a uniform 2D float array
skyline_array = np.zeros((len(processed_skyline), max_len))
for i, row in enumerate(processed_skyline):
    skyline_array[i, :len(row)] = row
    
RAW_PROFILO_PING360.SKYLINE = skyline_array
print('Ping360 profiling data loaded successfully.')

DVL data loaded successfully.
ECHOSOUNDER data loaded successfully.
IMU data loaded successfully.
Vertical Profilo (Seaking) profiling data loaded successfully.
Ping360 profiling data loaded successfully.


In [3]:
save('HDF5_DATA/1_RAW_DATA.h5', 'RAW_DVL', RAW_DVL)
save('HDF5_DATA/1_RAW_DATA.h5', 'RAW_ECHO', RAW_ECHO)
save('HDF5_DATA/1_RAW_DATA.h5', 'RAW_IMU_EULER', RAW_IMU_EULER)
save('HDF5_DATA/1_RAW_DATA.h5', 'RAW_IMU_QUAT', RAW_IMU_QUAT)
save('HDF5_DATA/1_RAW_DATA.h5', 'RAW_PROFILO_SEAKING', RAW_PROFILO_SEAKING)
save('HDF5_DATA/1_RAW_DATA.h5', 'RAW_PROFILO_PING360', RAW_PROFILO_PING360)

Saved to HDF5_DATA/1_RAW_DATA.h5 -> RAW_DVL
Saved to HDF5_DATA/1_RAW_DATA.h5 -> RAW_ECHO
Saved to HDF5_DATA/1_RAW_DATA.h5 -> RAW_IMU_EULER
Saved to HDF5_DATA/1_RAW_DATA.h5 -> RAW_IMU_QUAT
Saved to HDF5_DATA/1_RAW_DATA.h5 -> RAW_PROFILO_SEAKING
Saved to HDF5_DATA/1_RAW_DATA.h5 -> RAW_PROFILO_PING360
